In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv("Synthetic.csv")
df = df.drop(columns='Recycling Score')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Item Name               10000 non-null  object 
 1   Brand Name              10000 non-null  object 
 2   Device Condition        10000 non-null  object 
 3   Material Recovery Rate  10000 non-null  float64
 4   Device Type             10000 non-null  object 
 5   Year of Manufacturing   10000 non-null  int64  
 6   Cost of Recovery        10000 non-null  float64
 7   Gold (g)                10000 non-null  float64
 8   Silver (g)              10000 non-null  float64
 9   Copper (g)              10000 non-null  float64
 10  Palladium (g)           10000 non-null  float64
 11  Platinum (g)            10000 non-null  float64
dtypes: float64(7), int64(1), object(4)
memory usage: 937.6+ KB


In [3]:
df.head()

,Item Name,Brand Name,Device Condition,Material Recovery Rate,Device Type,Year of Manufacturing,Cost of Recovery,Gold (g),Silver (g),Copper (g),Palladium (g),Platinum (g)
0,MacBook Pro,HP,Fair,79.304496,Laptop,2024,20.243908,0.077098,0.748185,105.975069,0.029234,0.001942
1,Inkjet Printer,OnePlus,Scrap,71.676932,Printer,2012,23.744438,0.003812,0.040175,27.207231,0.001703,0.000254
2,OnePlus 9,Sony,Damaged,78.152978,Smartphone,2021,13.242702,0.023239,0.187582,12.773561,0.007891,0.000626
3,LED TV,OnePlus,Fair,84.174559,TV,2009,24.787277,0.013565,0.213740,60.657142,0.004553,0.000883
4,Blade Server,Lenovo,Fair,76.277635,Server,2010,45.738895,0.144414,1.746150,313.832660,0.054359,0.003090


In [4]:
df["Device Age"] = 2026 - df["Year of Manufacturing"]

In [5]:
condition_map = {
    "Excellent": 5,
    "Good": 4,
    "Fair": 3,
    "Damaged": 2,
    "Scrap": 1
}

df["Condition Score"] = df["Device Condition"].map(condition_map)

In [6]:
x = df[[
    'Brand Name',
    'Device Condition',
    'Device Type',
    'Year of Manufacturing',
    "Device Age",
    "Condition Score"
]]

y = df[
    ['Gold (g)', 'Silver (g)', 'Copper (g)',
     'Palladium (g)', 'Platinum (g)',
     'Material Recovery Rate',
     'Cost of Recovery']
]

In [7]:
categorical_cols = [
    'Brand Name',
    'Device Condition',
    'Device Type'
]

numeric_cols = [
    'Year of Manufacturing',
    'Device Age',
    'Condition Score'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

In [8]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.20,
    random_state=42
)

In [9]:
y_train_metals = y_train[
    ['Gold (g)', 'Silver (g)', 'Copper (g)', 'Palladium (g)', 'Platinum (g)']
]

y_test_metals = y_test[
    ['Gold (g)', 'Silver (g)', 'Copper (g)', 'Palladium (g)', 'Platinum (g)']
]

y_train_recovery = y_train['Material Recovery Rate']
y_test_recovery = y_test['Material Recovery Rate']

y_train_cost = y_train['Cost of Recovery']
y_test_cost = y_test['Cost of Recovery']

In [10]:
metal_model = Pipeline([
    ('prep', preprocessor),
    ('model', MultiOutputRegressor(
        RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    ))
])

In [11]:
metal_model.fit(x_train, y_train_metals)

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [12]:
metal_pred = metal_model.predict(x_test)

In [13]:
print("Metal Prediction R2 Score:", r2_score(y_test_metals, metal_pred))
print("Metal Prediction MAE:", mean_absolute_error(y_test_metals, metal_pred))

Metal Prediction R2 Score: 0.9822348237813158
Metal Prediction MAE: 1.1609617021457772


In [14]:
metal_names = y_test_metals.columns

for i, metal in enumerate(metal_names):
    r2 = r2_score(y_test_metals.iloc[:, i], metal_pred[:, i])
    mae = mean_absolute_error(y_test_metals.iloc[:, i], metal_pred[:, i])
    
    print(f"{metal}")
    print(f"R2 : {r2:.4f}")
    print(f"MAE: {mae:.6f}")
    print("-" * 30)

Gold (g)
R2 : 0.9845
MAE: 0.003963
------------------------------
Silver (g)
R2 : 0.9857
MAE: 0.045824
------------------------------
Copper (g)
R2 : 0.9909
MAE: 5.753281
------------------------------
Palladium (g)
R2 : 0.9791
MAE: 0.001613
------------------------------
Platinum (g)
R2 : 0.9710
MAE: 0.000128
------------------------------


In [15]:
def train_single_target(y_train, y_test, name, n_estimators, max_depth, min_samples_split, min_samples_leaf, random_state, n_jobs):
    model = Pipeline([
        ('prep',preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state,
            n_jobs=n_jobs
        ))
    ])

    model.fit(x_train, y_train)
    pred = model.predict(x_test)

    print(f"\n{name}:")
    print("R2:", r2_score(y_test, pred))
    print("MAE", mean_absolute_error(y_test, pred))

    return model

In [16]:
recovery_model = train_single_target(
    y_train_recovery,
    y_test_recovery,
    "Recovery Rate Model",
    n_estimators=500,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


Recovery Rate Model:
R2: 0.6810601218031249
MAE 3.331722868426815


In [17]:
cost_model = train_single_target(
    y_train_cost,
    y_test_cost,
    "Cost of Recovery Model",
    n_estimators=400,
    max_depth=15,
    min_samples_split=3,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)


Cost of Recovery Model:
R2: 0.9577173218171278
MAE 1.7433552911710002


In [18]:
import joblib

#joblib.dump(metal_model, r"C:\Users\HP\OneDrive\Desktop\Sem-6\Minor Project\Model saves\metal_model.joblib")
#joblib.dump(recovery_model, r"C:\Users\HP\OneDrive\Desktop\Sem-6\Minor Project\Model saves\recovery_model.joblib")
#joblib.dump(cost_model, r"C:\Users\HP\OneDrive\Desktop\Sem-6\Minor Project\Model saves\cost_model.joblib")

print("All models saved successfully")

All models saved successfully
